## Dataset

### Load data

In [79]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

### Organizing features

In [80]:
data_final = pd.read_csv('match_consider.csv')

# Apply EDA results
data_final['HomeMatch1EloDiff'] = data_final['HomeMatch1TeamElo'] - data_final['HomeMatch1OppElo']
data_final['AwayMatch1EloDiff'] = data_final['AwayMatch1TeamElo'] - data_final['AwayMatch1OppElo']
data_final['HomeMatch2EloDiff'] = data_final['HomeMatch2TeamElo'] - data_final['HomeMatch2OppElo']
data_final['AwayMatch2EloDiff'] = data_final['AwayMatch2TeamElo'] - data_final['AwayMatch2OppElo']
data_final['HomeMatch3EloDiff'] = data_final['HomeMatch3TeamElo'] - data_final['HomeMatch3OppElo']
data_final['AwayMatch3EloDiff'] = data_final['AwayMatch3TeamElo'] - data_final['AwayMatch3OppElo']
data_final['HomeMatch4EloDiff'] = data_final['HomeMatch4TeamElo'] - data_final['HomeMatch4OppElo']
data_final['AwayMatch4EloDiff'] = data_final['AwayMatch4TeamElo'] - data_final['AwayMatch4OppElo']
data_final['HomeMatch5EloDiff'] = data_final['HomeMatch5TeamElo'] - data_final['HomeMatch5OppElo']
data_final['AwayMatch5EloDiff'] = data_final['AwayMatch5TeamElo'] - data_final['AwayMatch5OppElo']

data_final.drop(
    ['Prev1HomeFouls', 'Prev3HomeFouls', 'Prev5HomeFouls', 'Prev1AwayFouls', 'Prev3AwayFouls', 'Prev5AwayFouls'],
    axis=1, inplace=True)

print('\nShape of data_final:', data_final.shape)
print('\n----------data_final----------')
print(data_final.head())
print('\n----------Information----------')
print(data_final.info())


Shape of data_final: (137447, 91)

----------data_final----------
   Target       Team  Opponent  MatchTime  HomeAway  HomeMatch1Points  \
0       0    Ipswich   Burnley  2246400.0         0                 0   
1       1  Leicester  Brighton  2246400.0         1                 0   
2       3      Luton   Preston  2246400.0         1                 1   
3       1   Millwall       QPR  2246400.0         1                 1   
4       0    Preston     Luton  2246400.0         0                 0   

   HomeMatch2Points  HomeMatch3Points  HomeMatch4Points  HomeMatch5Points  \
0                 1                 0                 3                 3   
1                 3                 0                 1                 3   
2                 1                 3                 1                 3   
3                 0                 0                 0                 1   
4                 1                 1                 0                 1   

   ...  HomeMatch1EloDiff  Away

## Deep learning model

### Features and target

In [81]:
y_text = data_final["Target"].map({3: "Win", 1: "Draw", 0: "Lose"})
le = LabelEncoder()
y = le.fit_transform(y_text)

X = data_final.drop(columns=["Target", "Team", "Opponent", "MatchTime"], errors="ignore")

### Feature ordering

In [82]:
def feature_sort_key(col: str):
    prefix_rank = 3
    if col.startswith("Home"):
        prefix_rank = 0
    elif col.startswith("Away"):
        prefix_rank = 1
    elif col.startswith("Prev"):
        prefix_rank = 2

    import re
    nums = re.findall(r"\d+", col)
    num = int(nums[0]) if nums else 999

    return (prefix_rank, num, col)

X = X.reindex(sorted(X.columns, key=feature_sort_key), axis=1)

### Train and test split

In [83]:
def rolling_window_split(data_len, n_splits, batch, train_test_ratio):
    splits = []
    start = 0
    step = int((data_len - batch) / (n_splits - 1))

    train_size = int(batch * train_test_ratio)
    test_size = batch - train_size

    while True:
        train_start = start
        train_end = start + train_size
        test_start = train_end
        test_end = train_end + test_size

        if test_end > data_len:
            break

        train_index = np.arange(train_start, train_end)
        test_index = np.arange(test_start, test_end)

        splits.append((train_index, test_index))
        start += step

    return splits

In [84]:
batch = min(50000, len(X))
splits = rolling_window_split(len(X), n_splits=3, batch=batch, train_test_ratio=0.8)

train_idx, test_idx = splits[-1]

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y[train_idx], y[test_idx]

print("Classes:", list(le.classes_))
print("Train/Test shape:", X_train.shape, X_test.shape)

Classes: ['Draw', 'Lose', 'Win']
Train/Test shape: (40000, 87) (10000, 87)


### Scaling

In [85]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_cnn = X_train_scaled[..., np.newaxis]
X_test_cnn = X_test_scaled[..., np.newaxis]

### CNN model definition

In [86]:
num_features = X_train_cnn.shape[1]
num_classes = len(le.classes_)

def build_tabular_cnn(num_features: int, num_classes: int):
    inp = layers.Input(shape=(num_features, 1))

    x = layers.Conv1D(filters=64, kernel_size=7, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(filters=128, kernel_size=5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(filters=128, kernel_size=3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_tabular_cnn(num_features, num_classes)
model.summary()

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)          │ (None, 87, 1)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_26 (Conv1D)                   │ (None, 87, 64)              │             512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_26               │ (None, 87, 64)              │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_26 (ReLU)                      │ (None, 87, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_16 (MaxPooling1D)      │ (None, 43, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_27 (Conv1D)                   │ (None, 43, 128)             │          41,088 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_27               │ (None, 43, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_27 (ReLU)                      │ (None, 43, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_17 (MaxPooling1D)      │ (None, 21, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_28 (Conv1D)                   │ (None, 21, 128)             │          49,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_28               │ (None, 21, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ re_lu_28 (ReLU)                      │ (None, 21, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d_10          │ (None, 128)                 │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_16 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_16 (Dense)                     │ (None, 128)                 │          16,512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_17 (Dropout)                 │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_17 (Dense)                     │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 109,059 (426.01 KB)

 Trainable params: 108,419 (423.51 KB)

 Non-trainable params: 640 (2.50 KB)

### Training model

In [87]:
cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
]

history = model.fit(
    X_train_cnn, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    callbacks=cb,
    verbose=1,
)

Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - accuracy: 0.4245 - loss: 1.0845 - val_accuracy: 0.3683 - val_loss: 1.0812 - learning_rate: 0.0010
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.4457 - loss: 1.0615 - val_accuracy: 0.3905 - val_loss: 1.0715 - learning_rate: 0.0010
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.4487 - loss: 1.0584 - val_accuracy: 0.4556 - val_loss: 1.0530 - learning_rate: 0.0010
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.4521 - loss: 1.0557 - val_accuracy: 0.4515 - val_loss: 1.0537 - learning_rate: 0.0010
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.4528 - loss: 1.0542 - val_accuracy: 0.4241 - val_loss: 1.0688 - learning_rate: 0.0010
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.4560 - loss: 1.0522 - val_accuracy: 0.4580 - val_loss: 1.0495 - learning_rate: 0.0010
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.4576 - loss: 1.

### Evaluating model

In [88]:
proba = model.predict(X_test_cnn)
y_pred = np.argmax(proba, axis=1)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

print("\n=== CNN Evaluation ===")
print(f"Accuracy: {acc:.4f}")
print(f"Macro-F1: {macro_f1:.4f}\n")

print("Confusion Matrix (rows=true, cols=pred):")
print(pd.DataFrame(cm, index=[f"True_{c}" for c in le.classes_], columns=[f"Pred_{c}" for c in le.classes_]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step

=== CNN Evaluation ===
Accuracy: 0.4568
Macro-F1: 0.3488

Confusion Matrix (rows=true, cols=pred):
           Pred_Draw  Pred_Lose  Pred_Win
True_Draw          0       1600      1030
True_Lose          0       2656      1018
True_Win           0       1784      1912

Classification Report:
              precision    recall  f1-score   support

        Draw       0.00      0.00      0.00      2630
        Lose       0.44      0.72      0.55      3674
         Win       0.48      0.52      0.50      3696

    accuracy                           0.46     10000
   macro avg       0.31      0.41      0.35     10000
weighted avg       0.34      0.46      0.39     10000



### Training model with margin rule

In [89]:
cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-5),
]

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("class_weight:", class_weight)

history = model.fit(
    X_train_cnn, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    callbacks=cb,
    class_weight=class_weight,
    verbose=1,
)

class_weight: {0: 1.2731149941118431, 1: 0.9046294411651628, 2: 0.901631953836444}
Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 37ms/step - accuracy: 0.4731 - loss: 1.0580 - val_accuracy: 0.4585 - val_loss: 1.0554 - learning_rate: 1.2500e-04
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 44ms/step - accuracy: 0.4718 - loss: 1.0535 - val_accuracy: 0.4569 - val_loss: 1.0528 - learning_rate: 1.2500e-04
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.4679 - loss: 1.0522 - val_accuracy: 0.4279 - val_loss: 1.0727 - learning_rate: 1.2500e-04
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 0.4619 - loss: 1.0522 - val_accuracy: 0.4364 - val_loss: 1.0764 - learning_rate: 1.2500e-04
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 0.4641 - loss: 1.0500 - val_accuracy: 0.4556 - val_loss: 1.0577 - learning_rate: 1.2500e-04
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.4669 - loss: 1.0490 - val_accuracy: 0.4408 - val_loss: 1.0583 - l

### Evaluating model with margin rule

In [90]:
proba = model.predict(X_test_cnn)
class_names = list(le.classes_)

i_draw = class_names.index("Draw")
i_lose = class_names.index("Lose")
i_win  = class_names.index("Win")

margin = 0.08
y_pred = np.argmax(proba, axis=1)

close_win_lose = np.abs(proba[:, i_win] - proba[:, i_lose]) < margin
min_draw_prob = 0.25
y_pred[close_win_lose & (proba[:, i_draw] > min_draw_prob)] = i_draw

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

print("\n=== CNN Evaluation ===")
print(f"Accuracy: {acc:.4f}")
print(f"Macro-F1: {macro_f1:.4f}\n")

print("Confusion Matrix (rows=true, cols=pred):")
print(pd.DataFrame(cm, index=[f"True_{c}" for c in le.classes_], columns=[f"Pred_{c}" for c in le.classes_]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step

=== CNN Evaluation ===
Accuracy: 0.4272
Macro-F1: 0.4214

Confusion Matrix (rows=true, cols=pred):
           Pred_Draw  Pred_Lose  Pred_Win
True_Draw        937        859       834
True_Lose       1199       1678       797
True_Win        1161        878      1657

Classification Report:
              precision    recall  f1-score   support

        Draw       0.28      0.36      0.32      2630
        Lose       0.49      0.46      0.47      3674
         Win       0.50      0.45      0.47      3696

    accuracy                           0.43     10000
   macro avg       0.43      0.42      0.42     10000
weighted avg       0.44      0.43      0.43     10000



In [69]:
def focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_oh = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0-1e-7)

        ce = -y_true_oh * tf.math.log(y_pred)
        weight = alpha * tf.pow(1 - y_pred, gamma)
        fl = weight * ce
        return tf.reduce_sum(fl, axis=-1)
    return loss

In [70]:
model = build_tabular_cnn(num_features, num_classes)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=focal_loss(gamma=2.0, alpha=0.5),
    metrics=["accuracy"]
)

history = model.fit(
    X_train_cnn, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    callbacks=cb,
    class_weight=class_weight,
    verbose=1
)


Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.3807 - loss: 0.2483 - val_accuracy: 0.3960 - val_loss: 0.2427 - learning_rate: 0.0010
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - accuracy: 0.4103 - loss: 0.2392 - val_accuracy: 0.4250 - val_loss: 0.2376 - learning_rate: 0.0010
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.4200 - loss: 0.2380 - val_accuracy: 0.4180 - val_loss: 0.2374 - learning_rate: 0.0010
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.4308 - loss: 0.2371 - val_accuracy: 0.4238 - val_loss: 0.2369 - learning_rate: 0.0010
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.4362 - loss: 0.2366 - val_accuracy: 0.4524 - val_loss: 0.2346 - learning_rate: 0.0010
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.4366 - loss: 0.2364 - val_accuracy: 0.4313 - val_loss: 0.2437 - learning_rate: 0.0010
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.4323 - loss: 0.

In [71]:
proba = model.predict(X_test_cnn)
y_pred = np.argmax(proba, axis=1)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", acc)
print("Macro-F1:", macro_f1)
print(cm)
print(classification_report(y_test, y_pred, target_names=le.classes_))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Accuracy: 0.4448
Macro-F1: 0.416587644012916
[[ 549 1094  987]
 [ 648 1990 1036]
 [ 635 1152 1909]]
              precision    recall  f1-score   support

        Draw       0.30      0.21      0.25      2630
        Lose       0.47      0.54      0.50      3674
         Win       0.49      0.52      0.50      3696

    accuracy                           0.44     10000
   macro avg       0.42      0.42      0.42     10000
weighted avg       0.43      0.44      0.43     10000



In [73]:
val_ratio = 0.2
n = X_train_cnn.shape[0]
cut = int(n * (1 - val_ratio))

X_tr, y_tr = X_train_cnn[:cut], y_train[:cut]
X_val, y_val = X_train_cnn[cut:], y_train[cut:]

model = build_tabular_cnn(num_features, num_classes)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=focal_loss(gamma=2.0, alpha=0.5),
              metrics=["accuracy"])
model.fit(X_tr, y_tr, epochs=50, batch_size=256, verbose=0,
          validation_data=(X_val, y_val), callbacks=cb)

val_proba = model.predict(X_val, verbose=0)

class_names = list(le.classes_)
i_draw = class_names.index("Draw")
i_lose = class_names.index("Lose")
i_win  = class_names.index("Win")

def apply_rule(proba, margin, t_draw, t_close):
    pred = np.argmax(proba, axis=1)

    pred[proba[:, i_draw] >= t_draw] = i_draw

    close = np.abs(proba[:, i_win] - proba[:, i_lose]) < margin
    pred[close & (proba[:, i_draw] >= t_close)] = i_draw
    return pred

best = None
margins = np.linspace(0.02, 0.20, 10)
t_draws = np.linspace(0.20, 0.55, 8)
t_closes = np.linspace(0.15, 0.45, 7)

for margin in margins:
    for t_draw in t_draws:
        for t_close in t_closes:
            pred = apply_rule(val_proba, margin, t_draw, t_close)
            score = f1_score(y_val, pred, average="macro")
            if (best is None) or (score > best["macro_f1"]):
                best = {"macro_f1": score, "margin": float(margin),
                        "t_draw": float(t_draw), "t_close": float(t_close)}

print("BEST on VAL:", best)

BEST on VAL: {'macro_f1': 0.42253888000845236, 'margin': 0.08000000000000002, 't_draw': 0.35000000000000003, 't_close': 0.30000000000000004}


In [74]:
test_proba = model.predict(X_test_cnn, verbose=0)
y_pred = apply_rule(test_proba, best["margin"], best["t_draw"], best["t_close"])
print("Macro-F1(test):", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred, target_names=le.classes_))

Macro-F1(test): 0.4138101660537699
              precision    recall  f1-score   support

        Draw       0.29      0.27      0.28      2630
        Lose       0.46      0.47      0.47      3674
         Win       0.49      0.51      0.50      3696

    accuracy                           0.43     10000
   macro avg       0.41      0.41      0.41     10000
weighted avg       0.43      0.43      0.43     10000

